# Βήμα 8 – Σύστημα Α: τελική αξιολόγηση
Το τελικό μοντέλο του Συστήματος Α: UNSW-NB15, **8 χαρακτηριστικά**, δίκτυο 256-128-64 με dropout 0.1 (notebook 10).
Εδώ **δεν γίνεται καμία εκπαίδευση**· φορτώνουμε το αποθηκευμένο δίκτυο και τον preprocessor του και μετράμε:

1. Συνεπής τελική απόφαση (επίθεση → τύπος → σοβαρότητα).
2. Πίνακες σύγχυσης για τύπο και σοβαρότητα.
3. Κατώφλι απόφασης, επιλεγμένο στο validation.
4. Risk score.
5. Πού πηγαίνουν τα λάθη.

Χρόνος: 1–2 λεπτά.

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

DATA, MODELS, FIG = "../data/", "../models/", "../figures/"

with open(MODELS + "unsw_meta.json", encoding="utf-8") as f:
    meta = json.load(f)
with open(MODELS + "unsw_8num_tuned_config.json", encoding="utf-8") as f:
    config = json.load(f)
ATTACK_CLASSES, SEVERITY_CLASSES = meta["attack_classes"], meta["severity_classes"]
SEV_WEIGHTS = np.array([meta["severity_weight"][c] for c in SEVERITY_CLASSES])
SEV_OF_ATTACK = np.array([SEVERITY_CLASSES.index(meta["severity_map"][c]) for c in ATTACK_CLASSES])
print("Τελικό μοντέλο:", config)

## 8.1 Δεδομένα και μοντέλο
Ίδιος διαχωρισμός με πριν. Ο preprocessor **φορτώνεται από το αρχείο** (δεν ξαναϋπολογίζεται): έτσι ελέγχουμε ότι αυτό που αποθηκεύσαμε
αρκεί για να ξαναβγούν τα αποτελέσματα – ακριβώς ό,τι θα κάνει και η συνάρτηση πρόβλεψης σε νέα δεδομένα.

In [ ]:
train_full = pd.read_csv(DATA + "UNSW_NB15_train-set.csv")
test_df = pd.read_csv(DATA + "UNSW_NB15_test-set.csv")
for df in (train_full, test_df):
    df["attack_cat"] = df["attack_cat"].str.strip()
    df["severity"] = df["attack_cat"].map(meta["severity_map"])
if meta["drop_duplicates"]:
    train_full = train_full.drop_duplicates(subset=[c for c in train_full.columns if c != "id"])
_, val_df = train_test_split(train_full, test_size=0.15, stratify=train_full["attack_cat"], random_state=meta["random_state"])
frames = {"val": val_df, "test": test_df}

def encode_targets(df):
    return {"label": df["label"].to_numpy("int64"),
            "attack_cat": df["attack_cat"].map({c: i for i, c in enumerate(ATTACK_CLASSES)}).to_numpy("int64"),
            "severity": df["severity"].map({c: i for i, c in enumerate(SEVERITY_CLASSES)}).to_numpy("int64")}
y = {s: encode_targets(df) for s, df in frames.items()}

preprocessor = joblib.load(MODELS + "unsw_preprocessor_8num_tuned.joblib")
model = keras.models.load_model(MODELS + "unsw_multitask_8num_tuned.keras", compile=False)   # μόνο για προβλέψεις
X = {s: preprocessor.transform(df).astype("float32") for s, df in frames.items()}

prob = {}
for s in frames:
    p_label, p_attack, p_sev = model.predict(X[s], batch_size=1024, verbose=0)
    prob[s] = {"label": p_label.ravel(), "attack": p_attack, "sev": p_sev}
print("Σχήματα:", X["val"].shape, X["test"].shape, "| παράμετροι δικτύου:", model.count_params())

## 8.2 Συνεπής τελική απόφαση
Ίδιος κανόνας με το notebook 06:
1. `P(επίθεση) ≥ κατώφλι` → επίθεση, αλλιώς Normal / None.
2. Αν επίθεση → τύπος = ο πιο πιθανός **τύπος επίθεσης**· σοβαρότητα = αυτή που αντιστοιχεί στον τύπο.

Ο έλεγχος στο τέλος πρέπει να δώσει τα νούμερα του notebook 10 (test: label acc 0.8583, attack_cat acc 0.7165, macro-F1 0.4511).

In [ ]:
def decide(p_label, p_attack, threshold=0.5):
    is_attack = p_label >= threshold
    attack = np.where(is_attack, p_attack[:, 1:].argmax(1) + 1, 0)     # 0 = Normal
    return is_attack.astype(int), attack, SEV_OF_ATTACK[attack]

def summary(split, threshold=0.5, consistent=True):
    p, t = prob[split], y[split]
    if consistent:
        lab, att, sev = decide(p["label"], p["attack"], threshold)
    else:
        lab, att, sev = (p["label"] >= threshold).astype(int), p["attack"].argmax(1), p["sev"].argmax(1)
    return {"label acc": accuracy_score(t["label"], lab),
            "label precision": precision_score(t["label"], lab, zero_division=0),
            "label recall": recall_score(t["label"], lab),
            "label F1": f1_score(t["label"], lab),
            "label FPR (ψευδείς συναγερμοί)": ((lab == 1) & (t["label"] == 0)).sum() / (t["label"] == 0).sum(),
            "label ROC-AUC": roc_auc_score(t["label"], p["label"]),
            "attack_cat acc": accuracy_score(t["attack_cat"], att),
            "attack_cat macro-F1": f1_score(t["attack_cat"], att, average="macro"),
            "severity acc": accuracy_score(t["severity"], sev),
            "severity macro-F1": f1_score(t["severity"], sev, average="macro")}

cmp_decision = pd.DataFrame({"Ανεξάρτητες έξοδοι": summary("test", consistent=False), "Συνεπής απόφαση": summary("test")}).round(4)
print(cmp_decision.to_string())

raw_att, raw_sev = prob["test"]["attack"].argmax(1), prob["test"]["sev"].argmax(1)
raw_lab = (prob["test"]["label"] >= 0.5).astype(int)
conflict = ((raw_lab == 0) != (raw_att == 0)) | (SEV_OF_ATTACK[raw_att] != raw_sev)
print(f"\nΡοές όπου οι τρεις ανεξάρτητες έξοδοι διαφωνούν μεταξύ τους: {conflict.mean():.1%}")

## 8.3 Πίνακες σύγχυσης (test, κατώφλι 0.5)
Κάθε γραμμή είναι ένας πραγματικός τύπος· τα ποσοστά δείχνουν πού τον κατατάσσει το σύστημα (κάθε γραμμή αθροίζει στο 100%).

In [ ]:
yt = y["test"]
lab, att, sev = decide(prob["test"]["label"], prob["test"]["attack"])
print(classification_report(yt["attack_cat"], att, labels=range(len(ATTACK_CLASSES)),
                            target_names=ATTACK_CLASSES, digits=3, zero_division=0))

def plot_cm(y_true, y_pred, names, title, fname, size):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(names)))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cmn[i, j]:.0%}\n{cm[i, j]}" for j in range(len(names))] for i in range(len(names))])
    fig, ax = plt.subplots(figsize=size)
    sns.heatmap(cmn, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=names, yticklabels=names, annot_kws={"size": 8}, ax=ax)
    ax.set_xlabel("Πρόβλεψη"); ax.set_ylabel("Πραγματικό"); ax.set_title(title)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.savefig(FIG + fname, dpi=200); plt.show()
    return cm

cm_attack = plot_cm(yt["attack_cat"], att, ATTACK_CLASSES, "Τύπος επίθεσης – τελικό μοντέλο, 8 χαρακτηριστικά (test)",
                    "29_unsw_final_cm_attack_cat.png", (9.5, 8))
cm_sev = plot_cm(yt["severity"], sev, SEVERITY_CLASSES, "Σοβαρότητα – τελικό μοντέλο, 8 χαρακτηριστικά (test)",
                 "30_unsw_final_cm_severity.png", (5.5, 4.6))

## 8.4 Κατώφλι απόφασης
Το κατώφλι 0.5 δεν είναι υποχρεωτικό: πιο ψηλά → λιγότεροι ψευδείς συναγερμοί αλλά χάνονται περισσότερες επιθέσεις.

**Κανόνας:** το κατώφλι διαλέγεται στο **validation** (εκεί που μεγιστοποιεί την ακρίβεια) και μετά απλώς αναφέρουμε τι δίνει στο test.
Το «καλύτερο κατώφλι για το test» το δείχνουμε μόνο για να φανεί η διαφορά των δύο συνόλων· **δεν το χρησιμοποιούμε**, γιατί θα ήταν επιλογή πάνω στα δεδομένα ελέγχου.

In [ ]:
def at_threshold(split, t):
    pred, true = prob[split]["label"] >= t, y[split]["label"]
    return {"accuracy": accuracy_score(true, pred), "precision": precision_score(true, pred, zero_division=0),
            "recall": recall_score(true, pred), "FPR": ((pred == 1) & (true == 0)).sum() / (true == 0).sum()}

ths = np.round(np.arange(0.05, 1.0, 0.05), 2)
curve = {s: pd.DataFrame([at_threshold(s, t) for t in ths], index=ths) for s in ("val", "test")}
T_VAL = float(curve["val"]["accuracy"].idxmax())
T_TEST = float(curve["test"]["accuracy"].idxmax())
print(f"Κατώφλι που μεγιστοποιεί την ακρίβεια στο validation: {T_VAL}  <- αυτό κρατάμε")
print(f"Κατώφλι που θα μεγιστοποιούσε την ακρίβεια στο test:   {T_TEST}  (μόνο για σύγκριση)")
print(f"Κέρδος ακρίβειας στο validation σε σχέση με το 0.5: {curve['val']['accuracy'].max() - curve['val'].loc[0.5, 'accuracy']:+.4f}\n")

rows = sorted(set([0.3, 0.5, 0.7, 0.9, T_VAL, T_TEST]))
thr_table = pd.concat({"validation": curve["val"], "test": curve["test"]}, axis=1).loc[rows].round(4)
thr_table.index.name = "κατώφλι"
print(thr_table.to_string())
thr_table.to_csv(MODELS + "unsw_final_thresholds.csv")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for s, name, ls in [("val", "validation", "--"), ("test", "test", "-")]:
    fpr, tpr, _ = roc_curve(y[s]["label"], prob[s]["label"])
    axes[0].plot(fpr, tpr, color="#2a78d6", lw=2, ls=ls, label=f"{name} (AUC = {roc_auc_score(y[s]['label'], prob[s]['label']):.3f})")
    axes[1].plot(ths, curve[s]["recall"], color="#2a78d6", lw=2, ls=ls, label=f"Recall – {name}")
    axes[1].plot(ths, curve[s]["FPR"], color="#eb6834", lw=2, ls=ls, label=f"Ψευδείς συναγερμοί (FPR) – {name}")
axes[0].plot([0, 1], [0, 1], color="gray", ls=":", lw=1, label="Τυχαία πρόβλεψη")
axes[0].set_xlabel("Ψευδείς συναγερμοί (FPR)"); axes[0].set_ylabel("Recall (TPR)"); axes[0].set_title("Καμπύλη ROC"); axes[0].legend()
axes[1].axvline(T_VAL, color="gray", ls=":", lw=1.4, label=f"Μέγιστη ακρίβεια στο validation: {T_VAL}")
axes[1].set_xlabel("Κατώφλι απόφασης"); axes[1].set_title("Επίδραση του κατωφλιού")
axes[1].legend(fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.17), frameon=False)
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(FIG + "31_unsw_final_roc_threshold.png", dpi=200, bbox_inches="tight"); plt.show()

## 8.5 Risk score
`Risk = 100 · P(επίθεση) · E[βάρος σοβαρότητας | επίθεση]`, με βάρη Low 0.33, Medium 0.66, High 1.0.
Ένας αριθμός από 0 έως 100 για κάθε ροή. Αν δουλεύει σωστά, πρέπει να **ανεβαίνει** όσο ανεβαίνει η πραγματική σοβαρότητα.

In [ ]:
def risk_score(p_label, p_sev):
    p = p_sev[:, 1:]
    return (100 * p_label * (p * SEV_WEIGHTS[1:]).sum(1) / np.clip(p.sum(1), 1e-9, None)).astype("float64")

risk = risk_score(prob["test"]["label"], prob["test"]["sev"])
risk_df = pd.DataFrame({"risk": risk, "σοβαρότητα": np.array(SEVERITY_CLASSES)[yt["severity"]]})
risk_stats = risk_df.groupby("σοβαρότητα")["risk"].describe(percentiles=[0.25, 0.5, 0.75]).loc[SEVERITY_CLASSES].round(1)
print(risk_stats.to_string())
rho = spearmanr(risk, SEV_WEIGHTS[yt["severity"]])[0]
print(f"\nΣυσχέτιση Spearman του risk score με την πραγματική σοβαρότητα: {rho:.3f}")
print(f"ROC-AUC του risk score ως ανιχνευτή επίθεσης: {roc_auc_score(yt['label'], risk):.4f}")

bands = pd.cut(risk, [-0.01, 25, 50, 75, 100], labels=["0–25", "25–50", "50–75", "75–100"])
band_table = (pd.crosstab(risk_df["σοβαρότητα"], bands, normalize="index").loc[SEVERITY_CLASSES] * 100).round(1)
band_table.columns.name = "risk score"
print("\nΠοσοστό (%) των ροών κάθε πραγματικής σοβαρότητας ανά ζώνη risk score:")
print(band_table.to_string())
risk_stats.to_csv(MODELS + "unsw_final_risk_score.csv")

SEV_COLORS = ["#b9d3f0", "#7fb0e6", "#4a8fdc", "#1f5fb0"]            # ένα χρώμα, από ανοιχτό σε σκούρο = αυξανόμενη σοβαρότητα
fig, ax = plt.subplots(figsize=(7.5, 4.4))
data = [risk[yt["severity"] == i] for i in range(len(SEVERITY_CLASSES))]
bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False, medianprops={"color": "#1f1f1d", "linewidth": 1.6})
for patch, color in zip(bp["boxes"], SEV_COLORS):
    patch.set_facecolor(color); patch.set_edgecolor("#3a3a38")
ax.set_xticks(range(1, len(SEVERITY_CLASSES) + 1))
ax.set_xticklabels([f"{c}\n(n = {len(d)})" for c, d in zip(SEVERITY_CLASSES, data)])
ax.set_xlabel("Πραγματική σοβαρότητα"); ax.set_ylabel("Risk score"); ax.set_ylim(-2, 102)
ax.set_title("Risk score ανά πραγματική σοβαρότητα (test)")
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(FIG + "32_unsw_final_risk_score.png", dpi=200); plt.show()

## 8.6 Πού πηγαίνουν τα λάθη
Για κάθε πραγματικό τύπο: πόσο συχνά ανιχνεύεται ως επίθεση, πόσο συχνά βγαίνει σωστός ο τύπος, και με ποιους τύπους μπερδεύεται.
Ένα λάθος στον τύπο δεν είναι πάντα λάθος στη σοβαρότητα (π.χ. Backdoor → Exploits: και τα δύο High).

In [ ]:
rows = []
for i, name in enumerate(ATTACK_CLASSES):
    m = yt["attack_cat"] == i
    row = cm_attack[i].astype(float); row[i] = 0
    top = [f"{ATTACK_CLASSES[j]} {cm_attack[i, j] / m.sum():.0%}" for j in np.argsort(row)[::-1][:2] if row[j] > 0]
    rows.append({"πλήθος": int(m.sum()),
                 "ανιχνεύεται ως επίθεση": np.nan if i == 0 else lab[m].mean(),
                 "σωστός τύπος": (att[m] == i).mean(),
                 "σωστή σοβαρότητα": (sev[m] == yt["severity"][m]).mean(),
                 "συχνότερα λάθη": ", ".join(top)})
errors = pd.DataFrame(rows, index=ATTACK_CLASSES)
print(errors.round(3).to_string())
errors.to_csv(MODELS + "unsw_final_errors_by_type.csv")

detected = (yt["label"] == 1) & (lab == 1)
print(f"\nΑπό τις επιθέσεις που ανιχνεύθηκαν ({detected.sum()}): σωστός τύπος {(att[detected] == yt['attack_cat'][detected]).mean():.1%}, "
      f"σωστή σοβαρότητα {(sev[detected] == yt['severity'][detected]).mean():.1%}")

## 8.7 Τελικός πίνακας του Συστήματος Α
Τα νούμερα που θα μπουν στο κείμενο: με το κατώφλι 0.5 και με το κατώφλι που επιλέχθηκε στο validation.

In [ ]:
final = pd.DataFrame({("validation", "κατώφλι 0.5"): summary("val"), ("validation", f"κατώφλι {T_VAL}"): summary("val", T_VAL),
                      ("test", "κατώφλι 0.5"): summary("test"), ("test", f"κατώφλι {T_VAL}"): summary("test", T_VAL)}).round(4)
print(final.to_string())
final.to_csv(MODELS + "unsw_final_evaluation.csv")